# Study 2: Ensemble Learning for Heart Disease Prediction

This notebook presents the code workflow for Study 2 of the PhD research project.

The notebook is organized into five main cells:

1. Data loading and train-test split  
2. SMOTEENN and baseline models  
3. Bagging ensemble models  
4. Boosting ensemble models  
5. Stacking ensemble models and final result table  

In [ ]:
# ============================================================
# Cell 1: Data loading, train-test split, and helper functions
# Study 2: Ensemble Learning for Heart Disease Prediction
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    BaggingClassifier,
    AdaBoostClassifier,
    StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

RANDOM_STATE = 42

# ------------------------------------------------------------
# 1) Load dataset
# ------------------------------------------------------------
# Before running on GitHub / another computer:
# 1. Put the dataset file in the Data/ folder, or
# 2. Change DATA_PATH to match your local file path.
DATA_PATH = "...\\Partition class 0 Five_5 cluster on cleaned_data_afterDuplicate_tranform_dummies_numerical_STD_drop_Outlier.csv"

df = pd.read_csv(DATA_PATH)

# Target variable
TARGET = "HeartDisease"

X = df.drop(columns=[TARGET])
y = df[TARGET]

# ------------------------------------------------------------
# 2) Train-test split
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Dataset shape:", df.shape)
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("\nTraining class distribution:")
print(y_train.value_counts())
print("\nTesting class distribution:")
print(y_test.value_counts())

# ------------------------------------------------------------
# 3) Helper function for model evaluation
# ------------------------------------------------------------
results = []

def evaluate_model(model_name, model, X_train_data, y_train_data, X_test_data=X_test, y_test_data=y_test):
    """
    Fit a model, evaluate it on the test set, and store the results.
    Primary metric: Recall.
    """
    model.fit(X_train_data, y_train_data)
    y_pred = model.predict(X_test_data)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test_data)[:, 1]
        auc = roc_auc_score(y_test_data, y_score)
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test_data)
        auc = roc_auc_score(y_test_data, y_score)
    else:
        auc = np.nan

    tn, fp, fn, tp = confusion_matrix(y_test_data, y_pred).ravel()

    row = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test_data, y_pred),
        "Precision": precision_score(y_test_data, y_pred, zero_division=0),
        "Recall": recall_score(y_test_data, y_pred, zero_division=0),
        "F1_score": f1_score(y_test_data, y_pred, zero_division=0),
        "AUC": auc,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

    results.append(row)
    return row

def show_results():
    """Display all model results sorted by Recall."""
    return (
        pd.DataFrame(results)
        .sort_values(by="Recall", ascending=False)
        .reset_index(drop=True)
    )


# Data balancing with SMOTEENN and baseline models

In [ ]:
from imblearn.combine import SMOTEENN

# ------------------------------------------------------------
# 1) Apply SMOTEENN only to the training data
# ------------------------------------------------------------
smoteenn = SMOTEENN(random_state=RANDOM_STATE)
X_res, y_res = smoteenn.fit_resample(X_train, y_train)

print("Before SMOTEENN:")
print(y_train.value_counts())

print("\nAfter SMOTEENN:")
print(pd.Series(y_res).value_counts())

# ------------------------------------------------------------
# 2) Define baseline models
# ------------------------------------------------------------
base_models = {
    "RF": RandomForestClassifier(
        n_estimators=73,
        max_depth=12,
        min_samples_split=2,
        random_state=0,
        bootstrap=False
    ),
    "LR": LogisticRegression(
        random_state=0,
        max_iter=1000
    ),
    "ET": ExtraTreesClassifier(
        n_estimators=100,
        random_state=0
    ),
    "SVM": SVC(
        probability=True,
        kernel="linear",
        random_state=0
    )
}

# ------------------------------------------------------------
# 3) Train and evaluate baseline models
# ------------------------------------------------------------
for name, model in base_models.items():
    evaluate_model(
        model_name=f"Baseline_{name}",
        model=clone(model),
        X_train_data=X_res,
        y_train_data=y_res
    )

show_results()


In [ ]:
# ============================================================
# Cell 3: Bagging ensemble models
# ============================================================

bagging_models = {
    f"Bagging_{name}": BaggingClassifier(
        estimator=clone(model),
        n_estimators=50,
        max_samples=0.8,
        max_features=1.0,
        bootstrap=True,
        random_state=RANDOM_STATE
    )
    for name, model in base_models.items()
}

for name, model in bagging_models.items():
    evaluate_model(
        model_name=name,
        model=model,
        X_train_data=X_res,
        y_train_data=y_res
    )

show_results()


In [ ]:
# ============================================================
# Cell 4: Boosting ensemble models
# ============================================================

boosting_models = {
    f"AdaBoost_{name}": AdaBoostClassifier(
        estimator=clone(model),
        n_estimators=50,
        learning_rate=1.0,
        random_state=RANDOM_STATE
    )
    for name, model in base_models.items()
}

for name, model in boosting_models.items():
    evaluate_model(
        model_name=name,
        model=model,
        X_train_data=X_res,
        y_train_data=y_res
    )

show_results()


In [ ]:
# ============================================================
# Cell 5: Stacking ensemble models and final result table
# ============================================================

stacking_combinations = {
    "Stacking_RF_LR": [
        ("RF", clone(base_models["RF"])),
        ("LR", clone(base_models["LR"]))
    ],
    "Stacking_RF_SVM": [
        ("RF", clone(base_models["RF"])),
        ("SVM", clone(base_models["SVM"]))
    ],
    "Stacking_RF_ET": [
        ("RF", clone(base_models["RF"])),
        ("ET", clone(base_models["ET"]))
    ],
    "Stacking_LR_SVM": [
        ("LR", clone(base_models["LR"])),
        ("SVM", clone(base_models["SVM"]))
    ],
    "Stacking_LR_ET": [
        ("LR", clone(base_models["LR"])),
        ("ET", clone(base_models["ET"]))
    ],
    "Stacking_SVM_ET": [
        ("SVM", clone(base_models["SVM"])),
        ("ET", clone(base_models["ET"]))
    ],
    "Stacking_RF_LR_SVM": [
        ("RF", clone(base_models["RF"])),
        ("LR", clone(base_models["LR"])),
        ("SVM", clone(base_models["SVM"]))
    ],
    "Stacking_RF_LR_ET": [
        ("RF", clone(base_models["RF"])),
        ("LR", clone(base_models["LR"])),
        ("ET", clone(base_models["ET"]))
    ],
    "Stacking_LR_SVM_ET": [
        ("LR", clone(base_models["LR"])),
        ("SVM", clone(base_models["SVM"])),
        ("ET", clone(base_models["ET"]))
    ],
    "Stacking_RF_SVM_ET": [
        ("RF", clone(base_models["RF"])),
        ("SVM", clone(base_models["SVM"])),
        ("ET", clone(base_models["ET"]))
    ],
    "Stacking_RF_LR_SVM_ET": [
        ("RF", clone(base_models["RF"])),
        ("LR", clone(base_models["LR"])),
        ("SVM", clone(base_models["SVM"])),
        ("ET", clone(base_models["ET"]))
    ]
}

for name, estimators in stacking_combinations.items():
    stacking_model = StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(max_iter=1000, random_state=0),
        n_jobs=-1
    )

    evaluate_model(
        model_name=name,
        model=stacking_model,
        X_train_data=X_res,
        y_train_data=y_res
    )

# ------------------------------------------------------------
# Final summary table
# ------------------------------------------------------------
final_results = show_results()
display(final_results)

# Save results for GitHub
os.makedirs("Results", exist_ok=True)
final_results.to_csv("Results/study2_ensemble_results.csv", index=False)

print("Saved result table to: Results/study2_ensemble_results.csv")
